# Semana 13 - Validação e Otimização de Modelos
### Atividade: GridSearch + Cross-Validation no dataset Titanic

Objetivo: aplicar `GridSearchCV` com validação cruzada em uma Árvore de Decisão, comparar com um modelo sem ajuste de hiperparâmetros e responder às perguntas do slide 17.

In [1]:
import kagglehub
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

path = kagglehub.dataset_download("yasserh/titanic-dataset")
df = pd.read_csv(f"{path}/Titanic-Dataset.csv")
df.head()

Using Colab cache for faster access to the 'titanic-dataset' dataset.


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 1. Pré-processamento

Colunas como `Name`, `Ticket` e `Cabin` são descartadas: a primeira e a segunda são identificadores praticamente únicos por linha, e `Cabin` tem um número muito alto de valores nulos. `Age` e `Embarked` são preenchidos em vez de descartados, já que têm poucos nulos e são variáveis relevantes.

In [2]:
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

X = df.drop(columns=['Survived'])
y = df['Survived']

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,3,0,22.0,1,0,7.2500,False,True
1,1,1,38.0,1,0,71.2833,False,False
2,3,1,26.0,0,0,7.9250,False,True
3,1,1,35.0,1,0,53.1000,False,True
4,3,0,35.0,0,0,8.0500,False,True


## 2. Divisão treino/teste (80/20)

O conjunto de teste é separado uma única vez e reservado até a avaliação final, para não influenciar a escolha de hiperparâmetros.

In [3]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Treino:', X_treino.shape)
print('Teste:', X_teste.shape)

Treino: (712, 8)
Teste: (179, 8)


## 3. Modelo padrão (sem ajuste de hiperparâmetros)

Serve como linha de base para medir se a otimização traz ganho real.

In [4]:
modelo_padrao = DecisionTreeClassifier(random_state=42)
modelo_padrao.fit(X_treino, y_treino)

acc_padrao = accuracy_score(y_teste, modelo_padrao.predict(X_teste))
print('Acurácia do modelo padrão:', round(acc_padrao, 4))

Acurácia do modelo padrão: 0.8212


Sem limite de profundidade, a árvore tende a crescer até separar quase todas as amostras de treino, o que costuma gerar overfitting: bom desempenho no treino, desempenho inferior no teste.

## 4. GridSearch + Cross-Validation

`cv=5` divide o treino em 5 dobras; cada combinação de hiperparâmetros é avaliada nas 5 e o resultado usado é a média.

In [5]:
param_grid = {
    'max_depth': [3, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy'
)
grid.fit(X_treino, y_treino)

print('Melhores hiperparâmetros:', grid.best_params_)
print('Melhor acurácia média (CV, treino):', round(grid.best_score_, 4))

Melhores hiperparâmetros: {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 5}
Melhor acurácia média (CV, treino): 0.8217


In [6]:
melhor_modelo = grid.best_estimator_
acc_otimizado = accuracy_score(y_teste, melhor_modelo.predict(X_teste))

print('Acurácia do modelo otimizado (teste):', round(acc_otimizado, 4))
print('Diferença (otimizado - padrão):', round(acc_otimizado - acc_padrao, 4))

Acurácia do modelo otimizado (teste): 0.7654
Diferença (otimizado - padrão): -0.0559


## 5. Experimento - variação do número de folds

Comparação de `cv=3`, `cv=5` e `cv=10`, mantendo os hiperparâmetros encontrados pelo GridSearch.

In [7]:
melhores_params = grid.best_params_

for cv_k in [3, 5, 10]:
    scores = cross_val_score(
        DecisionTreeClassifier(random_state=42, **melhores_params),
        X_treino, y_treino, cv=cv_k
    )
    print(f'cv={cv_k:2d} -> acurácias: {np.round(scores, 3)} | média: {scores.mean():.4f} | desvio: {scores.std():.4f}')

cv= 3 -> acurácias: [0.807 0.806 0.819] | média: 0.8104 | desvio: 0.0058
cv= 5 -> acurácias: [0.811 0.776 0.831 0.852 0.838] | média: 0.8217 | desvio: 0.0263
cv=10 -> acurácias: [0.778 0.806 0.817 0.761 0.789 0.845 0.873 0.831 0.761 0.887] | média: 0.8147 | desvio: 0.0423


Com mais folds, cada fold de teste fica menor, o que costuma aumentar a variância entre os resultados individuais, mesmo quando a média geral permanece parecida.

## 6. Experimento - GridSearch em outro modelo (Regressão Logística)

Verifica se o ganho da otimização é específico da árvore ou aparece em outro modelo também.

In [8]:
scaler = StandardScaler()
X_treino_s = scaler.fit_transform(X_treino)
X_teste_s = scaler.transform(X_teste)

lr_padrao = LogisticRegression(max_iter=1000, random_state=42)
lr_padrao.fit(X_treino_s, y_treino)
acc_lr_padrao = accuracy_score(y_teste, lr_padrao.predict(X_teste_s))

param_grid_lr = {'C': [0.01, 0.1, 1, 10, 100]}
grid_lr = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), param_grid_lr, cv=5)
grid_lr.fit(X_treino_s, y_treino)
acc_lr_otim = accuracy_score(y_teste, grid_lr.best_estimator_.predict(X_teste_s))

print('Regressão Logística - acurácia padrão:', round(acc_lr_padrao, 4))
print('Regressão Logística - melhor C:', grid_lr.best_params_, '-> acurácia otimizada:', round(acc_lr_otim, 4))

Regressão Logística - acurácia padrão: 0.8045
Regressão Logística - melhor C: {'C': 0.01} -> acurácia otimizada: 0.8045


## 7. Custo de tempo do GridSearch

In [9]:
t0 = time.time()
DecisionTreeClassifier(random_state=42).fit(X_treino, y_treino)
t1 = time.time()

GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5).fit(X_treino, y_treino)
t2 = time.time()

print('Tempo modelo padrão (s):', round(t1 - t0, 4))
print('Tempo GridSearch completo (s):', round(t2 - t1, 4))

Tempo modelo padrão (s): 0.014
Tempo GridSearch completo (s): 1.9309


A grade tem 4 × 3 × 2 = 24 combinações, cada uma avaliada em 5 folds, totalizando 120 treinos contra 1 único treino do modelo padrão. O tempo total costuma continuar baixo em datasets pequenos como este, mas cresce proporcionalmente ao tamanho da grade e ao custo de treino de cada modelo.

## Respostas (perguntas do slide 17)

*Preencher com os valores exatos obtidos após executar as células acima — os resultados podem variar levemente conforme a versão do dataset baixado.*

**1) Quais foram os melhores hiperparâmetros encontrados?**

Valores obtidos em `grid.best_params_` (combinação de `criterion`, `max_depth` e `min_samples_split`).

**2) O modelo otimizado teve uma acurácia melhor que o modelo padrão? Quanto?**

Comparar `acc_padrao` com `acc_otimizado`; a diferença indica se o ajuste de hiperparâmetros reduziu o overfitting observado no modelo padrão.

**3) O que aconteceu quando o número de folds no Cross-Validation foi alterado?**

Observar se a média das acurácias se mantém estável entre `cv=3`, `cv=5` e `cv=10`, e se o desvio padrão aumenta com mais folds (esperado, já que cada fold fica menor).

**4) O GridSearch compensa o tempo de processamento? Por quê?**

Comparar o tempo medido na célula 7 com o ganho de acurácia da célula 4. Para um dataset pequeno como o Titanic, o custo tende a ser baixo em termos absolutos; a resposta deve considerar se o ganho de acurácia justifica o número de treinos extras (24 combinações × 5 folds).